In [91]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [92]:
df = pd.read_csv("data/cleaned_diabetic_data.csv")
df.shape


(63595, 40)

In [93]:
df.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,...,miglitol,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,metformin-pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,[40-50),1,4,7,14,69,0,16,...,No,No,No,Down,No,No,No,Ch,Yes,0
1,Other,Female,[40-50),1,1,7,14,73,6,26,...,No,No,No,Up,No,No,No,Ch,Yes,0
2,Hispanic,Female,[50-60),1,6,7,14,69,0,25,...,No,No,No,Steady,No,No,No,No,Yes,0
3,AfricanAmerican,Male,[40-50),1,6,7,14,65,2,34,...,No,No,No,Up,No,No,No,Ch,Yes,0
4,Caucasian,Male,[50-60),3,1,1,14,63,4,30,...,No,No,No,Down,No,No,No,Ch,Yes,0


In [94]:
df.isnull().sum()

race                        0
gender                      0
age                         0
admission_type_id           0
discharge_disposition_id    0
admission_source_id         0
time_in_hospital            0
num_lab_procedures          0
num_procedures              0
num_medications             0
number_outpatient           0
number_emergency            0
number_inpatient            0
diag_1                      0
diag_2                      0
diag_3                      0
number_diagnoses            0
A1Cresult                   0
metformin                   0
repaglinide                 0
nateglinide                 0
chlorpropamide              0
glimepiride                 0
acetohexamide               0
glipizide                   0
glyburide                   0
tolbutamide                 0
pioglitazone                0
rosiglitazone               0
acarbose                    0
miglitol                    0
troglitazone                0
tolazamide                  0
insulin   

In [95]:
df.duplicated().sum()

np.int64(0)

In [96]:
df["readmitted"].value_counts()

readmitted
0    59058
1     4537
Name: count, dtype: int64

### Feature Engineering

In [97]:
df.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,...,miglitol,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,metformin-pioglitazone,change,diabetesMed,readmitted
0,Caucasian,Female,[40-50),1,4,7,14,69,0,16,...,No,No,No,Down,No,No,No,Ch,Yes,0
1,Other,Female,[40-50),1,1,7,14,73,6,26,...,No,No,No,Up,No,No,No,Ch,Yes,0
2,Hispanic,Female,[50-60),1,6,7,14,69,0,25,...,No,No,No,Steady,No,No,No,No,Yes,0
3,AfricanAmerican,Male,[40-50),1,6,7,14,65,2,34,...,No,No,No,Up,No,No,No,Ch,Yes,0
4,Caucasian,Male,[50-60),3,1,1,14,63,4,30,...,No,No,No,Down,No,No,No,Ch,Yes,0


In [98]:
df["diag_2"].unique()

array(['305', '599', '276', '250.82', '682', '507', '785', '300', 'V54',
       '707', 'V62', '780', '518', 'V45', '799', '440', '486', '428',
       '788', '436', '492', '789', '997', '427', '250.22', '263', '574',
       '250.6', '425', '517', '403', '153', '491', '198', '496', '998',
       '8', '416', '250', '285', '424', '493', '585', '584', '569', '426',
       '250.41', '250.7', '511', '414', '821', '995', '411', '415', '790',
       '721', '250.02', '38', '730', '41', '401', '197', 'V49', '648',
       '421', '438', '553', '850', '250.93', '453', '396', '852', '40',
       '576', '340', '596', '571', '196', '250.52', '410', '331', '431',
       '433', '531', '189', '711', '203', '560', '434', '253', '294',
       '157', '200', '250.01', '482', '781', '820', '578', '272', '185',
       '577', '728', '277', '286', '332', '797', '557', '787', '342',
       '296', '404', '304', '284', '311', '485', '515', '162', '535',
       '250.81', '250.03', '250.4', '348', '437', '996', '312',

In [99]:
def map_diagnosis_codes(code):
    code = str(code)
    if code.startswith("V") or code.startswith("E"):
        return "other"

    try:
        code_num = float(code)
    except ValueError:
        return "other"
    if 250 <= code_num < 251:
        return "Diabetes"
    elif (390 <=  code_num <= 459) or code_num == 785:
        return "Circulatory"
    elif (460 <= code_num <= 519) or code_num == 786:
        return "Respiratory"
    elif (520 <= code_num <= 579) or code_num == 787:
        return "Digestive"
    elif (580 <= code_num <= 629) or code_num == 788:
        return "Genitourinary"
    elif 800 <= code_num <= 999:
        return "Injury"
    elif 710 <= code_num <= 739:
        return "Musculoskeletal"
    elif 140 <= code_num <= 239:
        return "Neoplasms"
    else:
        return "other"
for col in ["diag_1","diag_2","diag_3"]:
    df[col] = df[col].apply(map_diagnosis_codes)

print(df["diag_1"].value_counts())

diag_1
Circulatory        19118
other              11426
Respiratory         8592
Digestive           6093
Diabetes            4793
Injury              4402
Musculoskeletal     3551
Genitourinary       3269
Neoplasms           2351
Name: count, dtype: int64


In [100]:
df["diag_1"].unique()

array(['other', 'Genitourinary', 'Circulatory', 'Diabetes', 'Respiratory',
       'Injury', 'Digestive', 'Neoplasms', 'Musculoskeletal'],
      dtype=object)

In [101]:
df["diag_2"].unique()

array(['other', 'Genitourinary', 'Diabetes', 'Respiratory', 'Circulatory',
       'Injury', 'Digestive', 'Neoplasms', 'Musculoskeletal'],
      dtype=object)

In [102]:
df["diag_3"].unique()

array(['Diabetes', 'Respiratory', 'Circulatory', 'Injury', 'Digestive',
       'other', 'Neoplasms', 'Genitourinary', 'Musculoskeletal'],
      dtype=object)

In [103]:
# Age
df["age"].unique()

array(['[40-50)', '[50-60)', '[60-70)', '[80-90)', '[30-40)', '[90-100)',
       '[70-80)', '[20-30)', '[10-20)', '[0-10)'], dtype=object)

In [104]:
age_order = [f"[{i}-{i+10})" for i in range(0, 100, 10)]
age_map = {age: i for i, age in enumerate(age_order)}
df["age"] = df["age"].map(age_map)

df["age"].unique()

array([4, 5, 6, 8, 3, 9, 7, 2, 1, 0])

In [105]:
df["age"].value_counts()

age
7    16148
6    14312
5    11269
8    10364
4     6089
3     2380
9     1684
2      961
1      329
0       59
Name: count, dtype: int64

In [106]:
df["total_prior_visits"] = df["number_outpatient"] + df["number_emergency"] + df["number_inpatient"]
print(df[["number_outpatient" , "number_emergency","number_inpatient","total_prior_visits"]].head())

   number_outpatient  number_emergency  number_inpatient  total_prior_visits
0                  0                 0                 0                   0
1                  0                 1                 0                   1
2                  0                 0                 1                   1
3                  1                 0                 1                   2
4                  0                 0                 0                   0


In [107]:
df.shape

(63595, 41)

### One_Hot_Encoding


In [108]:
df.head()

,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,num_lab_procedures,num_procedures,num_medications,...,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,metformin-pioglitazone,change,diabetesMed,readmitted,total_prior_visits
0,Caucasian,Female,4,1,4,7,14,69,0,16,...,No,No,Down,No,No,No,Ch,Yes,0,0
1,Other,Female,4,1,1,7,14,73,6,26,...,No,No,Up,No,No,No,Ch,Yes,0,1
2,Hispanic,Female,5,1,6,7,14,69,0,25,...,No,No,Steady,No,No,No,No,Yes,0,1
3,AfricanAmerican,Male,4,1,6,7,14,65,2,34,...,No,No,Up,No,No,No,Ch,Yes,0,2
4,Caucasian,Male,5,3,1,1,14,63,4,30,...,No,No,Down,No,No,No,Ch,Yes,0,0


In [109]:
df.dtypes.value_counts()

object    27
int64     14
Name: count, dtype: int64